# 04 - Shuffle, estrategias de join y skew

## Conexión con la teoría

Provocaremos un shuffle, compararemos sort-merge join y broadcast join y construiremos un
dataset desbalanceado para observar cómo una clave dominante concentra el trabajo.

In [ ]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("pontia-computacion-distribuida")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 1. Narrow frente a wide en el plan

In [ ]:
sales = (
    spark.range(0, 100_000, numPartitions=8)
    .withColumn("product_id", (F.col("id") % 100).cast("long"))
    .withColumn("amount", ((F.col("id") % 50) + 1).cast("double"))
)

narrow = sales.filter(F.col("amount") > 20).select("product_id", "amount")
wide = narrow.groupBy("product_id").agg(F.sum("amount").alias("revenue"))

print("PLAN NARROW")
narrow.explain(mode="formatted")
print("\nPLAN WIDE")
wide.explain(mode="formatted")

## 2. Join sin broadcast: ambas relaciones deben alinearse por clave

In [ ]:
products = spark.createDataFrame(
    [(i, f"product_{i}", f"family_{i % 5}") for i in range(100)],
    ["product_id", "product_name", "family"],
)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
regular_join = sales.join(products, "product_id")
regular_join.explain(mode="formatted")

Busca `SortMergeJoin`, `Exchange` y `Sort`. Hemos desactivado temporalmente el broadcast
automático para que la estrategia resulte visible.

## 3. Broadcast join: copiar la dimensión pequeña

In [ ]:
broadcast_join = sales.join(F.broadcast(products), "product_id")
broadcast_join.explain(mode="formatted")
broadcast_join.groupBy("family").agg(F.sum("amount").alias("revenue")).show()

`BroadcastHashJoin` evita redistribuir el lado grande, pero cada executor necesita una copia de
la tabla pequeña. No es adecuado si esa tabla no cabe con seguridad en memoria.

## 4. Crear skew de forma intencionada

In [ ]:
skewed = (
    spark.range(0, 100_000, numPartitions=8)
    .withColumn(
        "customer_id",
        F.when(F.col("id") < 90_000, F.lit("POPULAR"))
         .otherwise(F.concat(F.lit("customer_"), F.col("id") % 1_000)),
    )
    .withColumn("amount", F.lit(1.0))
)

skewed.groupBy("customer_id").count().orderBy(F.desc("count")).show(10)

## 5. Observar el desequilibrio después de particionar por clave

In [ ]:
by_customer = skewed.repartition(8, "customer_id")

distribution = (
    by_customer
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy(F.desc("count"))
)
distribution.show()

Una de las particiones recibe la mayoría de las filas porque una clave no puede repartirse
libremente si todas sus filas deben terminar juntas. En Spark UI esto aparece como una task
mucho más lenta, con más shuffle read y posiblemente spill.

## Posibles respuestas al skew

- Filtrar datos innecesarios antes del shuffle.
- Preagregar cuando sea posible.
- Utilizar broadcast si uno de los lados del join es realmente pequeño.
- Aplicar salting en casos concretos, asumiendo su complejidad.
- Permitir que Adaptive Query Execution divida particiones skewed cuando pueda.

No existe una receta universal: primero debemos medir y localizar la clave dominante.

## Preguntas de cierre

1. ¿Qué diferencias hay entre los planes de ambos joins?
2. ¿Qué dato se replica en el broadcast?
3. ¿Por qué aumentar el número de particiones no elimina por sí solo una clave dominante?
4. ¿Qué métricas buscarías en Spark UI para confirmar skew?

In [ ]:
spark.stop()